This is to convert our pure premium into a rate to charge each applicant 
using the formula : Idicated Average Rate = ( (Pure premium) + Fixed UW Expense per exposure ) / (1 - variable UW expense % - Underwiting profit %)
the values of UW expense per exposure, variable UW expense %, underwriting profit % , etc are from the "NAIC report on profitability by line by state in 2024" and we use the table on countrywide averages. From the table we used Homeowners Multiple Peril as it is most similar to our line of insurance

Now to estimate the pure premium for the "average person" we will use the sums of pure premiums of all exposures and incorporate the Loss adjustment expense (multiply by (1 + LAE/Losses)) 

Setup Cell:

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"
sys.path.insert(0, str(ROOT))

from src.rating import expected_cost, coefs, COVERAGES

freq = pd.read_csv(PROC / "frequency_data.csv").rename(columns={'class': 'class_year'})

Now to define the New pure premium including LAE

In [2]:
LAE_RATIO = 8.5 / 66.2        # NAIC 2024 IEE, Homeowners Multiple Peril

def loss_and_lae(risk):
    """Expected loss plus LAE for one policyholder, summed across coverages."""
    return expected_cost(risk) * (1 + LAE_RATIO)


students = freq.drop_duplicates('student_id')
L = students.apply(loss_and_lae, axis=1)

L_bar = L.mean()
print("mean loss + LAE per policy:", round(L_bar, 2))
print("actual loss per policy:    ", round(freq.total_amount.sum() / 10000, 2))

mean loss + LAE per policy: 1010.5
actual loss per policy:     851.47


We then ran the pricing algorithm on all students and found the Mean Pure premium per student

Now we estimate the Fixed underwriting expense by multiplying the mean premium(including LAE) by the 5.1% estimate of pure premium from the NAIC
1010.5 * 0.051 = 51.5355

This is a sample input to test the function

In [25]:
LAE_RATIO = 8.5 / 66.2          # NAIC 2024 IEE, Homeowners Multiple Peril
V         = (18.1 + 2.5) / 100  # variable expense %: selling + taxes, licenses, fees
E_F       = 1010.50 * 0.051     # fixed UW expense per exposure = 51.5355

Q_T       = 0.05               # target underwriting profit %  <- adjust here

risk = {'class_year':         'Junior',
        'study':              'Science',
        'gpa':                3.5,
        'greek':              'Non-greek',
        'off_campus':         'Off campus',
        'distance_to_campus': 1.2,
        'sprinklered':        False}

def indicated_rate(risk, e_f=E_F, v=V, q=Q_T, lae=LAE_RATIO):
    """Indicated rate for one policyholder, per Werner & Modlin ch. 8."""
    L = expected_cost(risk) * (1 + lae)      # pure premium including LAE
    return (L + e_f) / (1 - v - q)


print("Net premium (losses):", round(expected_cost(risk), 2))
print("Net premium + LAE:   ", round(expected_cost(risk) * (1 + LAE_RATIO), 2))
print("Gross premium:       ", round(indicated_rate(risk), 2))
print("Target profit on policy:", round(expected_cost(risk) * (1 + LAE_RATIO) * Q_T, 2))

Net premium (losses): 703.6
Net premium + LAE:    793.94
Gross premium:        1136.39
Target profit on policy: 39.7
